# 01 — NASA C-MAPSS: Full Dataset Understanding & EDA

**Project:** Industrial RUL Intelligence — pooled XGBoost model across FD001–FD004.

This notebook audits all four C-MAPSS subsets. They represent different operating-condition and fault-mode combinations, so the modelling notebook retains a subset indicator instead of pretending all rows come from one identical regime.

**Data boundary:** The training files contain complete run-to-failure trajectories; test files are truncated trajectories; each `RUL_FD00x.txt` contains the true RUL at the last observed cycle for each test engine. Official test labels are for final evaluation only. All RUL values are measured in **operating cycles**.

In [ ]:
from pathlib import Path
import zipfile
import shutil
import requests
import numpy as np
import pandas as pd

SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
COLUMN_NAMES = ["unit_nr", "time_cycles", "setting_1", "setting_2", "setting_3"] + [f"s_{i}" for i in range(1, 22)]
REQUIRED_FILES = [f"{kind}_{subset}.txt" for subset in SUBSETS for kind in ("train", "test", "RUL")]
DATASET_URLS = [
    "https://data.nasa.gov/docs/legacy/CMAPSSData.zip",
    "https://zenodo.org/records/15346912/files/CMAPSSData.zip?download=1",
    "https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip",
]

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CMAPSS_DIR = PROJECT_ROOT / "CMAPSSData"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
for folder in (RAW_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def find_dataset_file(filename):
    """Look in data/raw first, then an extracted CMAPSSData folder, then project root."""
    candidates = [RAW_DIR / filename, CMAPSS_DIR / filename, PROJECT_ROOT / filename]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def ensure_full_dataset(download_if_missing=True):
    missing = [name for name in REQUIRED_FILES if find_dataset_file(name) is None]
    if not missing:
        print("All 12 C-MAPSS files found across data/raw / CMAPSSData / project root.")
        return
    if not download_if_missing:
        raise FileNotFoundError(
            "Missing C-MAPSS files: " + ", ".join(missing) +
            ". Copy all train_FD00x.txt, test_FD00x.txt and RUL_FD00x.txt files into data/raw/ or extract CMAPSSData.zip into the project root."
        )
    archive_path = RAW_DIR / "CMAPSSData.zip"
    last_error = None
    downloaded = False
    for url in DATASET_URLS:
        try:
            print(f"Attempting to download the full C-MAPSS archive from {url.split('/')[2]} ...")
            temporary_path = archive_path.with_suffix(".zip.part")
            with requests.get(url, stream=True, timeout=(20, 180)) as response:
                response.raise_for_status()
                with open(temporary_path, "wb") as out:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            out.write(chunk)
            if not zipfile.is_zipfile(temporary_path):
                temporary_path.unlink(missing_ok=True)
                raise ValueError("The server response was not a valid ZIP archive.")
            temporary_path.replace(archive_path)
            downloaded = True
            break
        except Exception as exc:
            last_error = exc
            print(f"Download attempt failed: {type(exc).__name__}: {exc}")
    if not downloaded:
        raise RuntimeError(
            "Automatic download did not succeed. The full archive is available from the NASA PCoE repository: "
            "https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/ . "
            "Download '6. Turbofan Engine Degradation Simulation Data Set', extract it, and copy all 12 required .txt files to data/raw/. "
            f"Last error: {last_error}"
        )
    with zipfile.ZipFile(archive_path) as archive:
        members = {Path(member).name: member for member in archive.namelist() if not member.endswith("/")}
        not_in_archive = [name for name in REQUIRED_FILES if name not in members]
        if not_in_archive:
            raise FileNotFoundError(f"Archive is missing expected files: {not_in_archive}")
        for filename in REQUIRED_FILES:
            with archive.open(members[filename]) as src, open(RAW_DIR / filename, "wb") as dst:
                shutil.copyfileobj(src, dst)
    print(f"Extracted all C-MAPSS files to {RAW_DIR}")


def load_cmapss_file(filename):
    path = find_dataset_file(filename)
    if path is None:
        raise FileNotFoundError(f"{filename} was not found. Run ensure_full_dataset() or follow README.md.")
    frame = pd.read_csv(path, sep=r"\s+", header=None, names=COLUMN_NAMES, engine="python")
    frame = frame.dropna(axis=1, how="all")
    if frame.shape[1] != len(COLUMN_NAMES):
        raise ValueError(f"Unexpected column count in {path}: {frame.shape[1]}")
    return frame


def load_all_subsets(include_test=True):
    train_frames, test_frames, rul_frames = [], [], []
    for subset in SUBSETS:
        train = load_cmapss_file(f"train_{subset}.txt")
        train["subset"] = subset
        train["engine_key"] = subset + "_" + train["unit_nr"].astype(int).astype(str).str.zfill(3)
        train_frames.append(train)
        if include_test:
            test = load_cmapss_file(f"test_{subset}.txt")
            test["subset"] = subset
            test["engine_key"] = subset + "_" + test["unit_nr"].astype(int).astype(str).str.zfill(3)
            test_frames.append(test)
            rul_path = find_dataset_file(f"RUL_{subset}.txt")
            if rul_path is None:
                raise FileNotFoundError(f"RUL_{subset}.txt is missing.")
            rul_vector = pd.read_csv(rul_path, header=None).iloc[:, 0].to_numpy(dtype=float)
            test_ids = np.sort(test["unit_nr"].unique().astype(int))
            if len(test_ids) != len(rul_vector):
                raise ValueError(f"{subset}: {len(test_ids)} test engines but {len(rul_vector)} RUL labels.")
            rul_frames.append(pd.DataFrame({
                "subset": subset,
                "unit_nr": test_ids,
                "engine_key": [f"{subset}_{u:03d}" for u in test_ids],
                "true_rul_cycles": rul_vector,
            }))
    train_all = pd.concat(train_frames, ignore_index=True)
    if not include_test:
        return train_all
    return train_all, pd.concat(test_frames, ignore_index=True), pd.concat(rul_frames, ignore_index=True)


ensure_full_dataset(download_if_missing=True)
train_all, test_all, test_rul_all = load_all_subsets(include_test=True)

summary_rows = []
for subset in SUBSETS:
    tr = train_all.loc[train_all["subset"] == subset]
    te = test_all.loc[test_all["subset"] == subset]
    rl = test_rul_all.loc[test_rul_all["subset"] == subset, "true_rul_cycles"]
    summary_rows.append({
        "subset": subset,
        "train_engines": tr["engine_key"].nunique(),
        "train_rows": len(tr),
        "test_engines": te["engine_key"].nunique(),
        "test_rows": len(te),
        "official_test_rul_labels": len(rl),
        "train_median_life_cycles": float(tr.groupby("engine_key")["time_cycles"].max().median()),
        "test_median_last_observed_cycle": float(te.groupby("engine_key")["time_cycles"].max().median()),
    })
subset_summary = pd.DataFrame(summary_rows)
display(subset_summary)
print(f"All training rows: {len(train_all):,}")
print(f"Training engines: {train_all['engine_key'].nunique():,}")
print(f"All test rows: {len(test_all):,}")
print(f"Test engines / official test labels: {test_all['engine_key'].nunique():,} / {len(test_rul_all):,}")
print("Dataset files are resolved from data/raw/, then project-root/CMAPSSData/, then project root.")

## 1. Data-quality and schema audit

Each row contains an engine identifier, cycle number, three operating settings, and 21 sensor channels. Engine identifiers repeat between subsets, so the stable grouping key is `engine_key = subset + engine number`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_cols = ["unit_nr", "time_cycles", "setting_1", "setting_2", "setting_3"] + [f"s_{i}" for i in range(1, 22)]
audit_rows = []
for subset in SUBSETS:
    frame = train_all.loc[train_all["subset"] == subset]
    cycle_diff = frame.sort_values(["unit_nr", "time_cycles"]).groupby("unit_nr")["time_cycles"].diff().dropna()
    audit_rows.append({
        "subset": subset,
        "rows": len(frame),
        "engines": frame["engine_key"].nunique(),
        "missing_cells": int(frame[numeric_cols].isna().sum().sum()),
        "duplicate_rows": int(frame.duplicated(subset=["unit_nr", "time_cycles"]).sum()),
        "non_finite_cells": int((~np.isfinite(frame[numeric_cols].to_numpy(dtype=float))).sum()),
        "cycle_order_valid": bool((cycle_diff > 0).all()),
    })
display(pd.DataFrame(audit_rows))

## 2. Engine-life lengths across all subsets

The subsets have different operating conditions and engine trajectory lengths. This is one reason to keep the subset identity as a model feature and to report test metrics separately for FD001–FD004.

In [ ]:
train_life = train_all.groupby(["subset", "engine_key"], as_index=False)["time_cycles"].max().rename(columns={"time_cycles":"final_train_cycle"})
fig, ax = plt.subplots(figsize=(10, 5))
for subset in SUBSETS:
    values = train_life.loc[train_life["subset"] == subset, "final_train_cycle"].to_numpy()
    ax.hist(values, bins=22, histtype="step", linewidth=1.8, label=subset)
ax.set_title("Complete training-engine life lengths by subset")
ax.set_xlabel("Final operating cycle")
ax.set_ylabel("Number of engines")
ax.legend(title="Subset")
plt.tight_layout()
plt.show()
display(train_life.groupby("subset")["final_train_cycle"].describe().round(2))

## 3. Sensor behaviour and variation

Sensor channels can be nearly constant in one subset but vary in another because the operating regimes differ. For the pooled model we retain all 21 sensor channels and all three settings; XGBoost can ignore low-value features during tree construction.

In [ ]:
example_subset = "FD001"
example_key = train_all.loc[train_all["subset"] == example_subset, "engine_key"].iloc[0]
example = train_all.loc[train_all["engine_key"] == example_key].sort_values("time_cycles")
channels = ["s_2", "s_3", "s_7", "s_11", "s_12", "s_15"]
fig, axes = plt.subplots(3, 2, figsize=(13, 9), sharex=True)
for ax, sensor in zip(axes.flat, channels):
    ax.plot(example["time_cycles"], example[sensor], linewidth=1.2)
    ax.set_title(f"{example_subset} / {example_key}: {sensor}")
    ax.set_xlabel("Operating cycle")
    ax.set_ylabel("Sensor value")
plt.suptitle("Example run-to-failure sensor histories", y=1.01)
plt.tight_layout()
plt.show()

variability_rows = []
for subset in SUBSETS:
    sdev = train_all.loc[train_all["subset"] == subset, [f"s_{i}" for i in range(1, 22)]].std()
    for sensor, value in sdev.items():
        variability_rows.append({"subset": subset, "sensor": sensor, "std": float(value)})
variability = pd.DataFrame(variability_rows)
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=variability, x="sensor", y="std", hue="subset", ax=ax)
ax.set_title("Sensor variability by subset")
ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

## 4. Official test RUL references — descriptive only

These labels describe remaining life at each test engine's last observed cycle. Notebook 02 does not read them; Notebook 03 uses them for the final benchmark evaluation only.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for subset in SUBSETS:
    values = test_rul_all.loc[test_rul_all["subset"] == subset, "true_rul_cycles"].to_numpy()
    ax.hist(values, bins=20, histtype="step", linewidth=1.8, label=subset)
ax.set_title("Official test RUL labels by subset")
ax.set_xlabel("True RUL at last observed cycle (cycles)")
ax.set_ylabel("Number of engines")
ax.legend(title="Subset")
plt.tight_layout()
plt.show()
display(test_rul_all.groupby("subset")["true_rul_cycles"].describe().round(2))
print("Notebook 01 complete: all four subsets have been audited. Continue with Notebook 02.")

## Interview takeaways

- RUL is a regression target measured in cycles.
- The pooled problem spans different operating conditions and fault modes; a one-hot subset feature identifies the simulation regime.
- Unit numbers repeat across subsets, so validation groups must use subset + unit.
- Rolling features must use only the current and past cycles of the same engine.
- The official test RUL labels are not modelling inputs; they are reserved for final evaluation.